<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 Taller Práctico 08: Clasificación Binaria Médica
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Programación para Ciencia de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Taller Dummies • Módulo 08
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/homeworks/Para%20Dummies/08_Classification_Hands_On_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a practicar aquí? 🎯

Este taller guiado recorre el mismo camino que el **Taller Práctico 08**: entrenar un modelo que prediga si un cliente se va a ir (*churn*), evaluarlo con las métricas correctas, y compararlo contra un segundo modelo (k-NN).

Para no depender de descargar un archivo externo, vamos a **inventar** un dataset de clientes con las mismas características que tendría uno real — la lógica de cada paso es idéntica.


---
## Ejemplo resuelto: entrenar y evaluar en 5 líneas 🤖

Antes de meternos con el dataset completo de clientes, veamos el flujo completo de Scikit-Learn con un ejemplo diminuto: 20 clientes, una sola característica (antigüedad en meses), prediciendo si cancelan o no.


In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Datos de juguete: antigüedad en meses -> ¿canceló? (1 = sí, 0 = no)
antiguedad = np.array([[2], [3], [5], [8], [12], [18], [24], [30], [1], [4],
                        [6], [9], [15], [20], [26], [1], [3], [7], [11], [22]])
cancelo =    np.array([ 1,   1,   1,   0,    0,    0,    0,    0,   1,   1,
                         1,   0,    0,    0,    0,   1,   1,   0,    0,    0])

modelo = LogisticRegression().fit(antiguedad, cancelo)
predicciones = modelo.predict(antiguedad)

print("Exactitud (accuracy):", accuracy_score(cancelo, predicciones))
print("¿Predicción para un cliente de 1 mes?:", modelo.predict([[1]]))
print("¿Predicción para un cliente de 24 meses?:", modelo.predict([[24]]))


### 🤔 ¿Qué acaba de pasar?

- `LogisticRegression().fit(X, y)` entrena el modelo: le mostramos ejemplos con su respuesta correcta (`X` = antigüedad, `y` = si canceló) y él "aprende" el patrón.
- `.predict(...)` usa lo aprendido para adivinar la respuesta de datos nuevos.
- El patrón se repite siempre: **separar X e y → entrenar con `.fit()` → predecir con `.predict()` → medir qué tan bien le fue**.

Con esta misma receta vas a construir el pipeline completo del taller principal.


---
### 🟢 Paso 1 — Preparar los datos de clientes (Ejercicios 1.1, 1.2 y 2.1)

En el taller principal, `customer_churn.csv` ya viene con columnas como `contract`, `tenure` y `churn`. Aquí construimos una tabla equivalente a mano.

**Pistas:**
1. `pd.get_dummies(df, columns=['contrato'], drop_first=True)` convierte una columna de texto (categórica) en columnas de 0/1 — así el modelo puede usarla.
2. `X` son todas las columnas EXCEPTO la que quieres predecir; `y` es solo la columna objetivo (`churn`).
3. `train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)` separa en entrenamiento/prueba manteniendo la misma proporción de "sí/no" en ambos grupos (por eso `stratify=y`).

<details>
<summary>▶ Ver solución</summary>

```python
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

np.random.seed(42)
n = 300
df = pd.DataFrame({
    'antiguedad_meses': np.random.randint(1, 72, n),
    'uso_mensual_gb': np.random.uniform(1, 50, n).round(1),
    'quejas_soporte': np.random.poisson(1.2, n),
    'contrato': np.random.choice(['Mensual', 'Anual', 'Bianual'], n, p=[0.55, 0.30, 0.15])
})
# El churn depende (de forma simulada) de la antigüedad y del contrato
prob_churn = 0.7 - df['antiguedad_meses'] / 100 + (df['contrato'] == 'Mensual') * 0.2
df['churn'] = (np.random.random(n) < prob_churn.clip(0.05, 0.9)).astype(int)

print(df.head())
print("Proporción de churn:")
print(df['churn'].value_counts(normalize=True))

df_encoded = pd.get_dummies(df, columns=['contrato'], drop_first=True)
X = df_encoded.drop(columns=['churn'])
y = df_encoded['churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)
```
</details>


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

np.random.seed(42)
n = 300
df = pd.DataFrame({
    'antiguedad_meses': np.random.randint(1, 72, n),
    'uso_mensual_gb': np.random.uniform(1, 50, n).round(1),
    'quejas_soporte': np.random.poisson(1.2, n),
    'contrato': np.random.choice(['Mensual', 'Anual', 'Bianual'], n, p=[0.55, 0.30, 0.15])
})
prob_churn = 0.7 - df['antiguedad_meses'] / 100 + (df['contrato'] == 'Mensual') * 0.2
df['churn'] = (np.random.random(n) < prob_churn.clip(0.05, 0.9)).astype(int)

print(df.head())

# 1.2 One-Hot Encoding de 'contrato' + separar X, y

### TU CÓDIGO AQUÍ ###

# 2.1 train_test_split estratificado (75% entrenamiento / 25% prueba)

### TU CÓDIGO AQUÍ ###


---
### 🟡 Paso 2 — Entrenar la Regresión Logística y leer los Odds Ratios (Ejercicio 2.2)

**Pistas:**
1. `LogisticRegression(max_iter=1000, random_state=42).fit(X_train, y_train)` entrena el modelo.
2. Los coeficientes están en `modelo.coef_[0]`, uno por cada columna de `X`.
3. El **Odds Ratio** es simplemente `np.exp(coeficiente)`. Un valor mayor a 1 significa "aumenta el riesgo de churn"; menor a 1 significa "lo reduce".

<details>
<summary>▶ Ver solución</summary>

```python
from sklearn.linear_model import LogisticRegression

modelo_lr = LogisticRegression(max_iter=1000, random_state=42)
modelo_lr.fit(X_train, y_train)

tabla_odds = pd.DataFrame({
    'variable': X_train.columns,
    'coeficiente': modelo_lr.coef_[0],
    'odds_ratio': np.exp(modelo_lr.coef_[0])
}).sort_values('odds_ratio', ascending=False)

print(tabla_odds)
```
</details>


In [ ]:
### TU CÓDIGO AQUÍ ###


---
### 🟡 Paso 3 — Matriz de confusión y reporte de clasificación (Ejercicio 3.1)

**Pistas:**
1. `modelo_lr.predict(X_test)` da las clases (0/1); `modelo_lr.predict_proba(X_test)[:, 1]` da la probabilidad de churn.
2. `ConfusionMatrixDisplay.from_predictions(y_test, y_pred)` dibuja la matriz de confusión.
3. `classification_report(y_test, y_pred)` te da precisión, recall y F1 en un solo bloque de texto.

<details>
<summary>▶ Ver solución</summary>

```python
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

y_pred = modelo_lr.predict(X_test)
y_proba = modelo_lr.predict_proba(X_test)[:, 1]

ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title('Matriz de Confusión')
plt.show()

print(classification_report(y_test, y_pred))
```
</details>


In [ ]:
### TU CÓDIGO AQUÍ ###


---
### 🟠 Paso 4 — Curva ROC y AUC (Ejercicio 3.2)

**Pistas:**
1. `roc_curve(y_test, y_proba)` devuelve tres cosas: `fpr` (falsos positivos), `tpr` (verdaderos positivos) y los umbrales usados.
2. `roc_auc_score(y_test, y_proba)` te da un solo número entre 0.5 (modelo al azar) y 1.0 (modelo perfecto).
3. Grafica `fpr` en el eje X y `tpr` en el eje Y con `plt.plot(fpr, tpr)`.

<details>
<summary>▶ Ver solución</summary>

```python
from sklearn.metrics import roc_curve, roc_auc_score

fpr, tpr, umbrales = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

plt.plot(fpr, tpr, label=f'AUC = {auc:.2f}')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('Falsos Positivos (FPR)')
plt.ylabel('Verdaderos Positivos (TPR)')
plt.legend()
plt.show()
```
</details>


In [ ]:
### TU CÓDIGO AQUÍ ###


---
### 🟠 Paso 5 — Ajustar el umbral para priorizar el Recall (Ejercicio 3.3)

**Contexto:** por defecto, el modelo dice "churn" cuando la probabilidad supera 0.5. Pero si el negocio prefiere pecar de precavido (avisar a más clientes en riesgo aunque se equivoque más veces), hay que **bajar** ese umbral.

**Pistas:**
1. `precision_recall_curve(y_test, y_proba)` te da listas de precisión, recall y los umbrales asociados.
2. Busca el primer umbral donde el recall sea `>= 0.85`.
3. Vuelve a calcular las predicciones con: `(y_proba >= nuevo_umbral).astype(int)`.

<details>
<summary>▶ Ver solución</summary>

```python
from sklearn.metrics import precision_recall_curve

precision, recall, umbrales = precision_recall_curve(y_test, y_proba)

# Buscamos el umbral más alto que aún cumple recall >= 0.85
candidatos = umbrales[recall[:-1] >= 0.85]
nuevo_umbral = candidatos.max() if len(candidatos) > 0 else 0.5

y_pred_ajustado = (y_proba >= nuevo_umbral).astype(int)

print(f"Nuevo umbral: {nuevo_umbral:.2f}")
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_ajustado)
plt.title('Matriz de Confusión con umbral ajustado')
plt.show()
```
</details>


In [ ]:
### TU CÓDIGO AQUÍ ###


---
### 🔴 Paso 6 — Un segundo modelo: k-NN con `Pipeline` y `GridSearchCV` (Ejercicios 4.1 y 4.2)

**Pistas:**
1. Un `Pipeline` encadena pasos: primero `StandardScaler()` (porque k-NN es sensible a la escala de los números), luego `KNeighborsClassifier()`.
2. `GridSearchCV` prueba automáticamente varias combinaciones de hiperparámetros y se queda con la mejor, usando validación cruzada.
3. Para no esperar demasiado, usa una malla pequeña: `n_neighbors` entre 3 y 15 (impares), y `cv=5`.

<details>
<summary>▶ Ver solución</summary>

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

pipeline_knn = Pipeline([
    ('escalador', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

malla_parametros = {
    'knn__n_neighbors': [3, 5, 7, 9, 11, 13, 15],
    'knn__weights': ['uniform', 'distance']
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
busqueda = GridSearchCV(pipeline_knn, malla_parametros, cv=cv, scoring='roc_auc')
busqueda.fit(X_train, y_train)

print("Mejores hiperparámetros:", busqueda.best_params_)
print("Mejor ROC-AUC en validación:", busqueda.best_score_)
```
</details>


In [ ]:
### TU CÓDIGO AQUÍ ###


---
### 🔴 Paso 7 — Comparar los dos modelos y sacar tu conclusión (Ejercicios 5.1 y 5.2)

**Pistas:**
1. Repite `predict` y `predict_proba` con `busqueda.best_estimator_` (el mejor k-NN encontrado) sobre `X_test`.
2. Arma una tabla con `accuracy_score`, `precision_score`, `recall_score`, `f1_score` y `roc_auc_score` para cada modelo — una fila por modelo.
3. La conclusión no tiene una única respuesta "correcta": justifica tu elección pensando en qué le importa más al negocio, ¿precisión o recall?

<details>
<summary>▶ Ver solución</summary>

```python
from sklearn.metrics import precision_score, recall_score, f1_score

def evaluar(nombre, y_true, y_pred, y_proba):
    return {
        'Modelo': nombre,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1': f1_score(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_proba)
    }

mejor_knn = busqueda.best_estimator_
y_pred_knn = mejor_knn.predict(X_test)
y_proba_knn = mejor_knn.predict_proba(X_test)[:, 1]

tabla_comparativa = pd.DataFrame([
    evaluar('Regresión Logística', y_test, y_pred, y_proba),
    evaluar('k-NN (mejor combinación)', y_test, y_pred_knn, y_proba_knn)
])
print(tabla_comparativa)

# Conclusión del estudiante:
# Recomendaría el modelo con mayor Recall si el objetivo del negocio es
# no dejar pasar clientes que sí se van a ir, aunque eso implique
# contactar a algunos que en realidad no cancelarían.
```
</details>


In [ ]:
### TU CÓDIGO AQUÍ ###


---
## Resumen relámpago ⚡

| Código | Para qué sirve |
|---|---|
| `pd.get_dummies(df, columns=[...], drop_first=True)` | Convertir una columna categórica en columnas 0/1 |
| `train_test_split(X, y, stratify=y)` | Separar entrenamiento/prueba manteniendo la proporción de clases |
| `LogisticRegression().fit(X, y)` | Entrenar una regresión logística |
| `np.exp(coeficiente)` | Convertir un coeficiente en Odds Ratio |
| `classification_report(y_test, y_pred)` | Precisión, recall y F1 de un vistazo |
| `roc_curve` / `roc_auc_score` | Curva ROC y su área bajo la curva |
| `precision_recall_curve` | Encontrar un umbral que priorice recall o precisión |
| `Pipeline([...])` | Encadenar escalado + modelo en un solo objeto |
| `GridSearchCV(pipeline, malla, cv=...)` | Buscar automáticamente los mejores hiperparámetros |

➡️ **Siguiente paso:** vuelve al [Taller Práctico 08](../08_Classification_Hands_On.ipynb) y repite estos mismos pasos sobre el dataset real `customer_churn.csv` — ya tienes la receta completa.


---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos (Edición Para No Ingenieros)</i>
  </p>
</div>
